# Interopérabilité avec C avec le module `ctypes`

## Création du fichier C

In [1]:
%%file functions.c
double prod(double *x, int n) {
    double prod = 1.0;
    for(int i=0; i<n; i++) {
        prod *= x[i];
    }
    return prod;
}

void cumsum(double *a, double *b, int n) {
    b[0] = a[0];
    for(int i=1; i<n; i++) {
        b[i] = a[i] + b[i-1];
    }
}

Overwriting functions.c


## Compilation de la librairie partagée

In [2]:
!gcc -c -Wall -O2 -fPIC functions.c -o functions.o
!gcc -shared -o libfunctions.so functions.o

## Chargement de la librairie et définition des types des fonctions

In [3]:
import ctypes
import os
import platform
import numpy as np

libname = "./libfunctions.so"
if not os.path.exists(libname):
    raise FileNotFoundError(f"{libname} not found")

lib = ctypes.CDLL(libname)

lib.prod.argtypes = [ctypes.POINTER(ctypes.c_double), ctypes.c_int]
lib.prod.restype = ctypes.c_double

lib.cumsum.argtypes = [
    ctypes.POINTER(ctypes.c_double),
    ctypes.POINTER(ctypes.c_double),
    ctypes.c_int,
]
lib.cumsum.restype = None

## Définition des wrappers Python

In [4]:
def prod(x):
    arr = np.ascontiguousarray(x, dtype=np.float64)
    ptr = arr.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
    return lib.prod(ptr, len(arr))


def cumsum(a):
    a = np.ascontiguousarray(a, dtype=np.float64)
    b = np.empty_like(a)
    ptr_a = a.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
    ptr_b = b.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
    lib.cumsum(ptr_a, ptr_b, len(a))
    return b

## Utilisation des fonctions en Python

In [5]:
print(prod([1, 2, 3, 4]))
print(cumsum([1, 2, 3, 4]))

24.0
[ 1.  3.  6. 10.]


## Mesure du temps d'exécution

In [11]:
from timeit import timeit

print(
    timeit(
        setup="import numpy; a = numpy.linspace(-1, 1, 1_000_000)",
        stmt="cumsum(a)",
        number=10_000,
        globals={"cumsum": cumsum},
    )
)

17.086399685999822
